In [1]:
class Node:
    def __init__(self, node_type, label=None, feature=None):
        self.type = node_type      
        self.label = label         
        self.feature = feature     
        self.branches = {}         

In [2]:
def predict(tree, x):
    current = tree
    while current.type != "leaf":
        feature = current.feature
        value = x[feature]
        current = current.branches[value]
    return current.label

In [3]:

fail_leaf = Node("leaf", label="Fail")
pass_leaf = Node("leaf", label="Pass")

study_node = Node("internal", feature="Study")
study_node.branches["Low"] = fail_leaf
study_node.branches["High"] = pass_leaf

root = Node("internal", feature="Attendance")
root.branches["Low"] = fail_leaf
root.branches["High"] = study_node

student = {
    "Attendance": "High",
    "Study": "High"
}
print(predict(root, student))

Pass


In [4]:
import math

def entropy(labels):
    if len(labels) == 0:
        return 0
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    total = len(labels)
    result = 0
    for count in counts.values():
        p = count / total
        result -= p * math.log2(p)
    return result

In [5]:
dataset = [
    {"Attendance": "High", "Study": "High", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "High", "Study": "Low",  "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "High", "Study": "Low",  "Assignment": "No",  "Label": "Pass"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "Low",  "Study": "Low",  "Assignment": "No",  "Label": "Fail"},
    {"Attendance": "Low",  "Study": "Low",  "Assignment": "Yes", "Label": "Fail"},
    {"Attendance": "High", "Study": "High", "Assignment": "No",  "Label": "Pass"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "No",  "Label": "Fail"}
]

In [6]:
def conditional_entropy(dataset, feature, label_col="Label"):
    total = len(dataset)
    weighted_entropy = 0
    values = set(row[feature] for row in dataset)
    for value in values:
        subset = [
            row[label_col]
            for row in dataset
            if row[feature] == value
        ]
        weight = len(subset) / total
        weighted_entropy += weight * entropy(subset)
    return weighted_entropy

def mutual_information(dataset, feature, label_col="Label"):
    labels = [row[label_col] for row in dataset]
    h_y = entropy(labels)
    h_y_given_x = conditional_entropy(dataset, feature, label_col)
    return h_y - h_y_given_x

In [7]:
features = ["Attendance", "Study", "Assignment"]
for feature in features:
    mi = mutual_information(dataset, feature)
    print(feature, mi)

best_feature = max(
    features,
    key=lambda f: mutual_information(dataset, f)
)
print("Best feature:", best_feature)

Attendance 0.5487949406953987
Study 0.04879494069539858
Assignment 0.04879494069539858
Best feature: Attendance


In [8]:
from collections import Counter

def majority_label(dataset, label_col="Label"):
    labels = [row[label_col] for row in dataset]
    return Counter(labels).most_common(1)[0][0]

def train(dataset, features, label_col="Label"):
    labels = [row[label_col] for row in dataset]
    if len(set(labels)) == 1:
        return Node("leaf", label=labels[0])

    if len(features) == 0:
        return Node("leaf", label=majority_label(dataset, label_col))

    best_feature = max(
        features,
        key=lambda f: mutual_information(dataset, f, label_col)
    )

    node = Node("internal", feature=best_feature)

    values = set(row[best_feature] for row in dataset)
    remaining_features = [f for f in features if f != best_feature]

    for value in values:
        subset = [row for row in dataset if row[best_feature] == value]
        child = train(subset, remaining_features, label_col)
        node.branches[value] = child

    return node

In [9]:
features = ["Attendance", "Study", "Assignment"]
tree = train(dataset, features)

new_student = {
    "Attendance": "High",
    "Study": "Low",
    "Assignment": "No"
}
prediction = predict(tree, new_student)
print("Prediction:", prediction)

Prediction: Pass


In [10]:
# TASK 1
labels1 = ["Pass", "Pass", "Pass", "Pass"]
labels2 = ["Pass", "Pass", "Fail", "Fail"]
labels3 = ["Pass", "Fail", "Pass", "Fail"]

for name, labels in [("labels1", labels1), ("labels2", labels2), ("labels3", labels3)]:
    print(name, "->", entropy(labels))

labels1 -> 0.0
labels2 -> 1.0
labels3 -> 1.0


In [11]:
# TASK 2
for feature in ["Attendance", "Study", "Assignment"]:
    print(f"MI({feature}) = {mutual_information(dataset, feature)}")

MI(Attendance) = 0.5487949406953987
MI(Study) = 0.04879494069539858
MI(Assignment) = 0.04879494069539858


In [12]:
# TASK 3
test_students = [
    {"Attendance": "High", "Study": "High"},
    {"Attendance": "Low",  "Study": "High"},
    {"Attendance": "Low",  "Study": "Low"},
]

for s in test_students:
    print(s, "->", predict(root, s))

{'Attendance': 'High', 'Study': 'High'} -> Pass
{'Attendance': 'Low', 'Study': 'High'} -> Fail
{'Attendance': 'Low', 'Study': 'Low'} -> Fail


In [13]:
# TASK 4
features = ["Attendance", "Study", "Assignment"]
trained_tree = train(dataset, features)
print("Tree trained successfully. Root feature:", trained_tree.feature)

Tree trained successfully. Root feature: Attendance


In [14]:
# TASK 5
new_records = [
    {"Attendance": "High", "Study": "High", "Assignment": "Yes"},
    {"Attendance": "Low",  "Study": "Low",  "Assignment": "Yes"},
    {"Attendance": "High", "Study": "Low",  "Assignment": "No"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "Yes"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "No"},
]

for record in new_records:
    result = predict(trained_tree, record)
    print(record, "->", result)

{'Attendance': 'High', 'Study': 'High', 'Assignment': 'Yes'} -> Pass
{'Attendance': 'Low', 'Study': 'Low', 'Assignment': 'Yes'} -> Fail
{'Attendance': 'High', 'Study': 'Low', 'Assignment': 'No'} -> Pass
{'Attendance': 'Low', 'Study': 'High', 'Assignment': 'Yes'} -> Pass
{'Attendance': 'Low', 'Study': 'High', 'Assignment': 'No'} -> Fail
